# NLP Final Project: Arabic & English Text Classification

This notebook compares **traditional ML** and **transformer (BERT)** approaches for text classification.

## Tasks
1. **Arabic tweet sentiment** (Negative / Neutral / Positive)
2. **English mental-health status** classification

## Setup
1. Place datasets in `../data/` (see `../data/README.md`)
2. Install dependencies: `pip install -r ../requirements.txt`
3. Run cells top to bottom

> **Colab tip:** upload the CSVs, or mount Drive and update `DATA_DIR` below.


In [ ]:
# Paths work locally (repo root/data) and in Google Colab (/content)
from pathlib import Path

DATA_DIR = Path("../data")
if not DATA_DIR.exists():
    DATA_DIR = Path("/content")

ARABIC_CSV = DATA_DIR / "arabic_tweets.csv"
ENGLISH_CSV = DATA_DIR / "data.csv"

print("DATA_DIR:", DATA_DIR.resolve() if DATA_DIR.exists() else DATA_DIR)
print("Arabic CSV exists:", ARABIC_CSV.exists(), "->", ARABIC_CSV)
print("English CSV exists:", ENGLISH_CSV.exists(), "->", ENGLISH_CSV)


## 1. Install & imports

Skip the install cell if packages are already installed.


In [ ]:
# Uncomment if needed
# %pip install -q nltk pandas numpy scikit-learn gensim matplotlib seaborn emoji datasets transformers torch


In [ ]:
import re
import string
from pathlib import Path

import emoji
import matplotlib.pyplot as plt
import nltk
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from datasets import Dataset, DatasetDict
from gensim.models import Word2Vec
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import regexp_tokenize, word_tokenize
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
)
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
)

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)

sns.set(style="whitegrid")
print("Torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())


## 2. Arabic traditional ML

Preprocessing + TF-IDF / Bag-of-Words / Word2Vec with Logistic Regression, SVM, and MultinomialNB.


In [ ]:
arabic_stopwords = set(stopwords.words("arabic"))


def normalize_arabic(text: str) -> str:
    text = re.sub(r"[إأآا]", "ا", text)
    text = re.sub(r"ى", "ي", text)
    text = re.sub(r"ؤ", "و", text)
    text = re.sub(r"ئ", "ي", text)
    text = re.sub(r"ة", "ه", text)
    text = re.sub(r"[^\w\s]", "", text)
    text = re.sub(r"\d+", "", text)
    text = re.sub(r"[ـ]", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def tokenize_and_filter(text: str):
    tokens = regexp_tokenize(text, r"\w+")
    return [word for word in tokens if word not in arabic_stopwords]


df_ar = pd.read_csv(ARABIC_CSV)
df_ar = df_ar.dropna(subset=["Tweet_text", "Sentiment_label"])
df_ar["cleaned_text"] = df_ar["Tweet_text"].astype(str).apply(normalize_arabic)

X = df_ar["cleaned_text"]
y = df_ar["Sentiment_label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Arabic samples: {len(df_ar)} | train={len(X_train)} test={len(X_test)}")
print(y.value_counts())


In [ ]:
# Feature representations
tfidf = TfidfVectorizer(tokenizer=tokenize_and_filter, token_pattern=None)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

bow = CountVectorizer(tokenizer=tokenize_and_filter, token_pattern=None)
X_train_bow = bow.fit_transform(X_train)
X_test_bow = bow.transform(X_test)

tokenized_train = [tokenize_and_filter(doc) for doc in X_train]
tokenized_test = [tokenize_and_filter(doc) for doc in X_test]

w2v_model = Word2Vec(
    sentences=tokenized_train, vector_size=100, window=5, min_count=1, workers=4
)


def avg_w2v(tokens, model, vector_size=100):
    vectors = [model.wv[word] for word in tokens if word in model.wv]
    if not vectors:
        return np.zeros(vector_size)
    return np.mean(vectors, axis=0)


X_train_w2v = np.array([avg_w2v(t, w2v_model) for t in tokenized_train])
X_test_w2v = np.array([avg_w2v(t, w2v_model) for t in tokenized_test])

results = []


def evaluate(name, model, Xtr, Xte):
    model.fit(Xtr, y_train)
    preds = model.predict(Xte)
    results.append(
        {
            "Representation": name,
            "Model": type(model).__name__,
            "Accuracy": accuracy_score(y_test, preds),
            "F1": f1_score(y_test, preds, average="weighted"),
        }
    )


for rep_name, Xtr, Xte in [
    ("TF-IDF", X_train_tfidf, X_test_tfidf),
    ("BoW", X_train_bow, X_test_bow),
]:
    evaluate(rep_name, LogisticRegression(max_iter=1000), Xtr, Xte)
    evaluate(rep_name, SVC(), Xtr, Xte)
    evaluate(rep_name, MultinomialNB(), Xtr, Xte)

evaluate("Word2Vec", LogisticRegression(max_iter=1000), X_train_w2v, X_test_w2v)
evaluate("Word2Vec", SVC(), X_train_w2v, X_test_w2v)

results_df = pd.DataFrame(results).sort_values(by="F1", ascending=False).reset_index(drop=True)
print(results_df)

plt.figure(figsize=(10, 6))
sns.barplot(data=results_df, x="Representation", y="F1", hue="Model", palette="Set2")
plt.title("F1 Score by Text Representation and Model")
plt.ylim(0, 1.0)
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 6))
sns.barplot(data=results_df, x="Representation", y="Accuracy", hue="Model", palette="Set1")
plt.title("Accuracy by Text Representation and Model")
plt.ylim(0, 1.0)
plt.tight_layout()
plt.show()


### Expected traditional results (from project runs)

| Rank | Representation | Model | Accuracy | F1 |
|------|----------------|-------|----------|-----|
| 1 | TF-IDF | LogisticRegression | ~0.771 | ~0.766 |
| 2 | BoW | LogisticRegression | ~0.767 | ~0.765 |
| 3 | TF-IDF | SVC | ~0.766 | ~0.761 |

Word2Vec average embeddings scored lower (~0.65–0.68) on this short-text dataset.


## 3. Arabic transformer (AraBERT)

Fine-tune `aubmindlab/bert-base-arabertv02-twitter` for 3-class sentiment.


In [ ]:
def remove_urls(text):
    return re.sub(r"http\S+|www\S+|https\S+", "", text, flags=re.MULTILINE)


def remove_mentions(text):
    return re.sub(r"@\w+", "", text)


def remove_hashtags(text):
    return re.sub(r"#", "", text)


def remove_emojis(text):
    return emoji.replace_emoji(text, replace="")


def remove_tatweel(text):
    return re.sub(r"ـ", "", text)


def remove_arabic_punctuation(text):
    arabic_punctuations = '''`÷×؛<>_()*&^%][ـ،/:"؟.,'{}~¦+|!”…“–ـ'''
    return text.translate(str.maketrans("", "", arabic_punctuations))


def normalize_arabic_bert(text):
    text = re.sub("[إأآا]", "ا", text)
    text = re.sub("ى", "ي", text)
    text = re.sub("ؤ", "ء", text)
    text = re.sub("ئ", "ء", text)
    text = re.sub("ة", "ه", text)
    return text


def clean_arabic_for_bert(text):
    text = str(text)
    text = remove_urls(text)
    text = remove_mentions(text)
    text = remove_hashtags(text)
    text = remove_emojis(text)
    text = remove_tatweel(text)
    text = remove_arabic_punctuation(text)
    text = normalize_arabic_bert(text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


df_bert = pd.read_csv(ARABIC_CSV)
df_bert = df_bert.dropna(subset=["Sentiment_label", "Tweet_text"])
df_bert["cleaned_text"] = df_bert["Tweet_text"].apply(clean_arabic_for_bert)

label2id_ar = {"Negative": 0, "Neutral": 1, "Positive": 2}
id2label_ar = {v: k for k, v in label2id_ar.items()}
df_bert["label"] = df_bert["Sentiment_label"].map(label2id_ar)

train_df_ar, val_df_ar = train_test_split(
    df_bert, test_size=0.1, stratify=df_bert["label"], random_state=42
)
print(df_bert["Sentiment_label"].value_counts())
print(f"train={len(train_df_ar)} val={len(val_df_ar)}")


In [ ]:
model_name_ar = "aubmindlab/bert-base-arabertv02-twitter"
tokenizer_ar = AutoTokenizer.from_pretrained(model_name_ar)


def preprocess_ar(examples):
    return tokenizer_ar(
        examples["cleaned_text"],
        truncation=True,
        padding="max_length",
        max_length=128,
    )


train_dataset_ar = Dataset.from_pandas(train_df_ar.reset_index(drop=True))
val_dataset_ar = Dataset.from_pandas(val_df_ar.reset_index(drop=True))

train_dataset_ar = train_dataset_ar.map(preprocess_ar, batched=True)
val_dataset_ar = val_dataset_ar.map(preprocess_ar, batched=True)

train_dataset_ar.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
val_dataset_ar.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

datasets_ar = DatasetDict({"train": train_dataset_ar, "validation": val_dataset_ar})

model_ar = AutoModelForSequenceClassification.from_pretrained(
    model_name_ar,
    num_labels=3,
    id2label=id2label_ar,
    label2id=label2id_ar,
)


In [ ]:
def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=1)
    labels = p.label_ids
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="weighted"
    )
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "f1": f1, "precision": precision, "recall": recall}


training_args_ar = TrainingArguments(
    output_dir="./arabic_sentiment_finetuned",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=4,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    logging_dir="./logs_ar",
    logging_steps=50,
    seed=42,
    save_total_limit=2,
    report_to="none",
)

trainer_ar = Trainer(
    model=model_ar,
    args=training_args_ar,
    train_dataset=datasets_ar["train"],
    eval_dataset=datasets_ar["validation"],
    processing_class=tokenizer_ar,
    compute_metrics=compute_metrics,
)

trainer_ar.train()
results_ar = trainer_ar.evaluate()
print("Arabic BERT evaluation:", results_ar)

trainer_ar.save_model("./arabic_sentiment_finetuned_final")
tokenizer_ar.save_pretrained("./arabic_sentiment_finetuned_final")


In [ ]:
label_names_ar = ["Negative", "Neutral", "Positive"]
predictions_ar = trainer_ar.predict(val_dataset_ar)
preds_ar = np.argmax(predictions_ar.predictions, axis=1)

cm = confusion_matrix(val_dataset_ar["label"], preds_ar)
cm = cm.astype("float") / cm.sum(axis=1)[:, np.newaxis]

plt.figure(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    cmap="Blues",
    fmt=".2f",
    xticklabels=label_names_ar,
    yticklabels=label_names_ar,
)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Arabic Sentiment — Confusion Matrix")
plt.tight_layout()
plt.show()

print(classification_report(val_dataset_ar["label"], preds_ar, target_names=label_names_ar))


### Expected AraBERT results

- **Accuracy / F1 ≈ 0.84**
- Stronger than the best traditional model (~0.77 F1)


## 4. English transformer (BERT)

Fine-tune `bert-base-uncased` on mental-health status labels from `data.csv` (`statement`, `status`).


In [ ]:
stop_words_en = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()


def clean_english(text):
    if pd.isnull(text):
        return ""
    text = str(text).lower()
    text = text.translate(str.maketrans("", "", string.punctuation))
    tokens = word_tokenize(text)
    return " ".join(
        lemmatizer.lemmatize(word) for word in tokens if word not in stop_words_en
    )


df_en = pd.read_csv(ENGLISH_CSV, on_bad_lines="skip")
df_en["cleaned_statement"] = df_en["statement"].apply(clean_english)

unique_labels = sorted(df_en["status"].dropna().unique())
label2id_en = {label: idx for idx, label in enumerate(unique_labels)}
id2label_en = {idx: label for label, idx in label2id_en.items()}
df_en = df_en.dropna(subset=["cleaned_statement", "status"])
df_en["label"] = df_en["status"].map(label2id_en)

train_df_en, val_df_en = train_test_split(
    df_en, test_size=0.1, stratify=df_en["label"], random_state=42
)
print("Labels:", unique_labels)
print(f"English samples: {len(df_en)} | train={len(train_df_en)} val={len(val_df_en)}")


In [ ]:
model_name_en = "bert-base-uncased"
tokenizer_en = AutoTokenizer.from_pretrained(model_name_en)

train_dataset_en = Dataset.from_pandas(
    train_df_en[["cleaned_statement", "label"]].reset_index(drop=True)
)
val_dataset_en = Dataset.from_pandas(
    val_df_en[["cleaned_statement", "label"]].reset_index(drop=True)
)


def preprocess_en(examples):
    texts = [str(t) if t is not None else "" for t in examples["cleaned_statement"]]
    return tokenizer_en(texts, truncation=True, padding="max_length", max_length=128)


train_dataset_en = train_dataset_en.map(
    preprocess_en, batched=True, remove_columns=["cleaned_statement"]
)
val_dataset_en = val_dataset_en.map(
    preprocess_en, batched=True, remove_columns=["cleaned_statement"]
)

model_en = AutoModelForSequenceClassification.from_pretrained(
    model_name_en,
    num_labels=len(label2id_en),
    id2label=id2label_en,
    label2id=label2id_en,
)

training_args_en = TrainingArguments(
    output_dir="./english_emotion_model",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=4,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_dir="./logs_en",
    logging_steps=50,
    seed=42,
    save_total_limit=2,
    report_to="none",
)

trainer_en = Trainer(
    model=model_en,
    args=training_args_en,
    train_dataset=train_dataset_en,
    eval_dataset=val_dataset_en,
    processing_class=tokenizer_en,
    compute_metrics=compute_metrics,
)

trainer_en.train()
results_en = trainer_en.evaluate()
print("English BERT evaluation:", results_en)

trainer_en.save_model("./english_emotion_model_final")
tokenizer_en.save_pretrained("./english_emotion_model_final")


In [ ]:
predictions_en = trainer_en.predict(val_dataset_en)
preds_en = np.argmax(predictions_en.predictions, axis=1)

cm_en = confusion_matrix(val_dataset_en["label"], preds_en)
cm_en = cm_en.astype("float") / cm_en.sum(axis=1)[:, np.newaxis]

plt.figure(figsize=(10, 8))
sns.heatmap(
    cm_en,
    annot=True,
    fmt=".2f",
    cmap="Blues",
    xticklabels=unique_labels,
    yticklabels=unique_labels,
)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("English Status — Confusion Matrix")
plt.tight_layout()
plt.show()

print(classification_report(val_dataset_en["label"], preds_en, target_names=unique_labels))


### Expected English BERT results

- **Accuracy / F1 ≈ 0.81**
- Classes include Anxiety, Bipolar, Depression, Normal, Personality disorder, Stress, Suicidal

## Summary

| Approach | Task | Best F1 (approx.) |
|----------|------|-------------------|
| TF-IDF + LogReg | Arabic sentiment | 0.77 |
| AraBERT | Arabic sentiment | **0.84** |
| BERT-base | English status | **0.81** |

Transformers outperform classic sparse features on the Arabic sentiment task in this project.
